# Finite-ground CPW — HFSS Eigenmode

Inspect the finite-ground CPW’s eigenmodes. See the [eigenmode
requirements](../../../../simulation/requirements/eigenmode.html). The
public coupon is 500 µm long, with a 10 µm signal, 6 µm gaps and 80 µm
grounds. This current-PDK case selects Al 100 nm on Si 500 µm,
explicitly replacing the prior Nb sheet model. It has no wave/lumped
ports. The body-first facade uses native mesh behavior; it does not
expose the prior CPW `LengthMeshSpec` operation.

Fresh construction uses the local shared-physical candidate, not a
published wheel. Retained results keep their original runtime
identities.

Published historical baseline:
[SCGSim](https://github.com/OrPenStrike/scgsim/tree/c9270c7e76e0b2df187dddde462079e00d410dd0).

## Setup and Imports

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path
from uuid import uuid4

import gdsfactory as gf
from IPython.display import display
from scgsim.aedt import (
    AedtResources,
    EigenmodeRunControl,
    EigenmodeSim,
    resolve_results,
)
from scgsim.geometry import ApproximationProfile, GeometryPlan

import orcapen_sc_pdk
from orcapen_sc_pdk.cells import finite_ground_cpw_coupon
from orcapen_sc_pdk.materials import get_material_records
from orcapen_sc_pdk.tech import get_single_die_layer_stack

## Setup and Run Controls

In [ ]:
WORKFLOW_ACTION = "run"  # AUTHORING_ONLY | prepare_handoff | prepare_only | run | analyze_handoff
RUN_ID = f"cpw_hfss_eigenmode_{uuid4().hex}"
OUTPUT_ROOT = Path.cwd() / ".artifacts" / "SCGSimExamples" / "CpwFiniteGround"
RUN_DIR = OUTPUT_ROOT / RUN_ID
RETURNED_RUN_DIR = None  # Set independently to an immutable returned-run directory.
SCGSIM_RUNTIME_SOURCE = "LOCAL_SHARED_PHYSICAL_CANDIDATE_UNCOMMITTED"
RESOURCES = AedtResources(cores=4, ram_limit_percent=30)
MODELING = "solid"
APPROXIMATION_PROFILE = ApproximationProfile(name=MODELING, revision="v1", parameters={})
# CPW trace length along X (um).
trace_length_um = 500.0
# Signal conductor width (um).
signal_width_um = 10.0
# Gap from signal to each ground conductor (um).
gap_um = 6.0
# Width of each ground conductor (um).
ground_width_um = 80.0
# AEDT version used for project generation.
aedt_version = "2024.2"
# AEDT project name.
project_name = RUN_ID
# AEDT design name.
design_name = "CpwFiniteGroundEigenmode"
# Padding around the model in -X, +X, -Y, +Y, -Z, +Z directions (um).
region_padding_um = (100.0, 100.0, 100.0, 100.0, 1000.0, 1000.0)
# Minimum search frequency (GHz), requested modes, pass limit, and convergence delta (%).
# Minimum eigenfrequency search bound (GHz).
minimum_frequency_ghz = 3.0
# Number of eigenmodes requested.
num_modes = 2
# Maximum adaptive passes.
maximum_passes = 6
# Pass convergence threshold on frequency change (%).
maximum_delta_frequency_percent = 5.0

## Create Simulation Component / Coupon

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    orcapen_sc_pdk.activate()
    gf.clear_cache()
    coupon = finite_ground_cpw_coupon(
        trace_length_um=trace_length_um,
        signal_width_um=signal_width_um,
        gap_um=gap_um,
        ground_width_um=ground_width_um,
    )
    layer_stack = get_single_die_layer_stack()
    material_records = get_material_records()

## Initialize AEDT Project / App

In [ ]:
# Only the generated native runner owns an AEDT session.

## Import GDS and Build the HFSS/Q3D/Q2D Model

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    plan = GeometryPlan(
        coupon,
        root_id="cpw",
        layer_stack=layer_stack,
        material_records=material_records,
        coupon_padding_um=0.0,
    )
    plan.set_nets(
        {
            "Signal": ("cpw/SIGNAL",),
            "GroundPlus": ("cpw/GROUND_PLUS",),
            "GroundMinus": ("cpw/GROUND_MINUS",),
        }
    )
    plan.set_vacuum_region(
        padding={
            "x_minus_um": region_padding_um[0],
            "x_plus_um": region_padding_um[1],
            "y_minus_um": region_padding_um[2],
            "y_plus_um": region_padding_um[3],
            "z_minus_um": region_padding_um[4],
            "z_plus_um": region_padding_um[5],
        }
    )
    snapshot = plan.prepare()

## Geometry Verification

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    display(coupon)
    display({"bbox_um": str(coupon.dbbox())})

## Materials and Boundaries

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    display(
        {
            level_id: {
                "material": layer_stack.layers[level_id].material,
                "zmin_um": layer_stack.layers[level_id].zmin,
                "thickness_um": layer_stack.layers[level_id].thickness,
            }
            for level_id in ("D0_TOP_M1", "D0_SUBSTRATE")
        }
    )

## Ports / Nets / Excitations

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    sim = EigenmodeSim(modeling=APPROXIMATION_PROFILE.name)
    sim.set_plan(snapshot)
    sim.set_output_dir(RUN_DIR)
    # No wave or lumped excitations in this Eigenmode case.

## Simulation Setup

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    run_control = EigenmodeRunControl(
        "Setup1", minimum_frequency_ghz, num_modes, maximum_passes, maximum_delta_frequency_percent
    )
    sim.set_eigenmode(
        project_name=project_name,
        design_name=design_name,
        run_control=run_control,
        resources=RESOURCES,
    )

## Simulation Configuration

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    display(sim.input_summary())

## Solve and Export

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "prepare_only", "run"}:
    HANDOFF = sim.prepare_handoff()
    display(HANDOFF)
    if WORKFLOW_ACTION == "run":
        subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Prepare-only native model inspection

Select `prepare_only` to build, read back, save and release the native
model without Analyze. This does not produce solved physics results.

In [ ]:
if WORKFLOW_ACTION == "prepare_only":
    from scgsim.aedt import resolve_prepared_model

    subprocess.run(
        [
            sys.executable,
            "-m",
            "scgsim.aedt.run",
            "--handoff",
            str(HANDOFF.metadata_path),
            "--execute",
            "--prepare-only",
        ],
        check=True,
    )
    display(resolve_prepared_model(HANDOFF.run_dir))

## Adaptive-Pass Convergence / Solver Diagnostics

In [ ]:
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    result = resolve_results(
        HANDOFF.run_dir if WORKFLOW_ACTION == "run" else Path(RETURNED_RUN_DIR)
    )
    result.show_all_results(show_details=True)

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    display(result.physics_results())

### Simulation Performance / Benchmarks

In [ ]:
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    display(result.simulation_benchmark())

## Save and Release AEDT

In [ ]:
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    display({"project": result.project_path, "receipt": result.receipt_path})
elif WORKFLOW_ACTION in {"prepare_handoff", "prepare_only", "run"}:
    display(HANDOFF.archive_path)